In [1]:
!pip install transformers datasets scikit-learn openai accelerate


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset
from scipy.special import softmax

/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
train_df = pd.read_csv("/Users/nili/PycharmProjects/MasterProject/exist2025/Dataset/train.csv")
dev_df = pd.read_csv("/Users/nili/PycharmProjects/MasterProject/exist2025/Dataset/val.csv")
test_df = pd.read_csv("/Users/nili/PycharmProjects/MasterProject/exist2025/Dataset/test.csv")

print(train_df.head())
print(train_df.shape)

   test_case      id language  \
0  EXIST2023  101705       es   
1  EXIST2021   11108       es   
2  EXIST2021   10405       es   
3  EXIST2021    8658       en   
4  EXIST2023  200886       en   

                                                text  \
0  segundo: el mundo, y ecuador en especial, es u...   
1  @vox_es dicen que las mujeres son valientes pe...   
2  @elisacarrio jubilate, gorda termotanque de ca...   
3  you guys don’t understand the struggle of have...   
4  boris johnson never brushing his hair shows ge...   

                                        task1  \
0  ['YES', 'YES', 'YES', 'YES', 'YES', 'YES']   
1                                      sexist   
2                                      sexist   
3                                  non-sexist   
4     ['YES', 'YES', 'NO', 'NO', 'NO', 'YES']   

                                               task2  hard_label_task1  \
0  [['MISOGYNY-NON-SEXUAL-VIOLENCE'], ['STEREOTYP...                 1   
1                    

In [4]:
#Preprocessing

In [5]:
df = pd.concat([train_df, dev_df])

train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["hard_label_task2"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [6]:
def convert_to_dataset(df):

    df = df.copy()

    df["text"] = df["text"].fillna("").astype(str)
    df["hard_label_task2"] = df["hard_label_task2"].astype(int)

    data = {
        "text": df["text"].tolist(),
        "label": df["hard_label_task2"].tolist()
    }

    return Dataset.from_dict(data)

train_dataset = convert_to_dataset(train_df)
val_dataset = convert_to_dataset(val_df)

In [7]:
#For test
test_dataset = Dataset.from_dict({
    "text": test_df["text"].tolist()
})

In [8]:
from transformers import AutoTokenizer
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast= False)


In [9]:
MAX_LENGTH = 128

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize)
val_dataset = val_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

Map:   0%|          | 0/13972 [00:00<?, ? examples/s]

Map:   0%|          | 0/2466 [00:00<?, ? examples/s]

Map:   0%|          | 0/1827 [00:00<?, ? examples/s]

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5 

)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted")
    }

In [12]:
training_args = TrainingArguments(

    output_dir="./model_task2_hard",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
   
    num_train_epochs=4,

    weight_decay=0.01,

    save_strategy="no", 
    load_best_model_at_end=True,

    logging_dir="./logs",

    report_to=None
)

In [13]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    compute_metrics=compute_metrics
)

/var/folders/w2/7wgmb9zn01d8k6wv97czc7th0000gn/T/ipykernel_96391/434736182.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [14]:
#!pip install -U accelerate

In [15]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,0.995700
1000,0.780300
1500,0.679800
2000,0.597300
2500,0.542300
3000,0.476400


TrainOutput(global_step=3496, training_loss=0.6436572871437335, metrics={'train_runtime': 1936.8476, 'train_samples_per_second': 28.855, 'train_steps_per_second': 1.805, 'total_flos': 3676352614674432.0, 'train_loss': 0.6436572871437335, 'epoch': 4.0})

In [16]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.778464674949646, 'eval_accuracy': 0.6650446066504461, 'eval_f1_macro': 0.4869894863609181, 'eval_f1_weighted': 0.6378297397664937, 'eval_runtime': 29.7013, 'eval_samples_per_second': 83.027, 'eval_steps_per_second': 5.219, 'epoch': 4.0}


In [17]:
pred = trainer.predict(val_dataset)

logits = pred.predictions
labels = pred.label_ids

probs = softmax(logits, axis=1)

predictions = np.argmax(probs, axis=1)
confidence = np.max(probs, axis=1)

print(classification_report(labels, predictions))

/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


              precision    recall  f1-score   support

           0       0.78      0.84      0.81      1338
           1       0.63      0.62      0.62       286
           2       0.46      0.53      0.49       275
           3       0.41      0.49      0.45       182
           4       0.50      0.61      0.55       176
           5       0.00      0.00      0.00       209

    accuracy                           0.67      2466
   macro avg       0.46      0.51      0.49      2466
weighted avg       0.61      0.67      0.64      2466



/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

In [18]:
from openai import OpenAI

client = OpenAI(api_key="xxx")

def query_llm(text):
    prompt = f"""You are an expert linguistic annotator. Classify the following sexist tweet into exactly ONE category.

Categories:
0: IDEOLOGICAL AND INEQUALITY (Discredits feminism, rejects inequality, or portrays men as victims)
1: STEREOTYPING AND DOMINANCE (Women as inferior, suitable only for specific roles, or "impossible to understand")
2: OBJECTIFICATION (Women as objects, focus on physical attributes/clothing, or comparison to services)
3: SEXUAL VIOLENCE (Sexual suggestions, harassment, or references to assault)
4: MISOGYNY AND NON-SEXUAL VIOLENCE (General hatred of women, calling them "toxic" or "poison")

### Examples:
Tweet: "Feminism is just a way for women to get special treatment." -> Category: 0
Tweet: "Women are naturally more emotional and less logical." -> Category: 1
Tweet: "She's just a 10/10 body with a 0/10 brain." -> Category: 2
Tweet: "I'd love to see her react if someone grabbed her like that." -> Category: 3
Tweet: "Women are a cancer to society nowadays." -> Category: 4

### Task:
Tweet: "{text}"
Return ONLY the category number (0, 1, 2, 3, or 4)."""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=1
        )
        return int(response.choices[0].message.content.strip())
    except Exception as e:
        print(f"Error: {e}")
        return 4 # Default to Misogyny if it fails

In [19]:
THRESHOLD = 0.65

texts = val_df["text"].tolist()

final_predictions = []


for i in range(len(predictions)):

    if confidence[i] >= THRESHOLD:

        final_predictions.append(predictions[i])

    else:

        llm_pred = query_llm(texts[i])

        final_predictions.append(llm_pred)

In [20]:
print(
    classification_report(
        labels,
        final_predictions
    )
)

              precision    recall  f1-score   support

           0       0.76      0.85      0.80      1338
           1       0.52      0.52      0.52       286
           2       0.47      0.41      0.44       275
           3       0.44      0.41      0.42       182
           4       0.34      0.53      0.41       176
           5       0.00      0.00      0.00       209

    accuracy                           0.63      2466
   macro avg       0.42      0.45      0.43      2466
weighted avg       0.58      0.63      0.60      2466



/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

In [21]:
test_pred = trainer.predict(test_dataset)

test_logits = test_pred.predictions

test_probs = softmax(test_logits, axis=1)

test_preds = np.argmax(test_probs, axis=1)
test_conf = np.max(test_probs, axis=1)

/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [22]:
test_texts = test_df["text"].tolist()

final_test_predictions = []

for i in range(len(test_preds)):

    if test_conf[i] >= THRESHOLD:

        final_test_predictions.append(test_preds[i])

    else:

        llm_pred = query_llm(test_texts[i])

        final_test_predictions.append(llm_pred)

In [23]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "label": final_test_predictions
})

submission.to_csv("submission_hard_2.csv", index=False)